## SOTA RAG with EquoAI!

In [1]:
! pip install equoai 
! pip install PyPDF2
! pip install sentence-transformers
! pip install ollama 
! pip install requests 

In [ ]:
from equoai import equonode as EquoNode
from sentence_transformers import SentenceTransformer
import os 
import PyPDF2
import numpy as np
from ollama import Client
import sys
import requests

In [72]:
def query_with_ollama(prompt : str, is_verbose=True) -> str:
    '''Query function using Ollama proxy'''

    # client = Client(host=os.getenv("LOCAL_SERVER_URI"))
    client = Client(host="https://ollama.newatlantis.top")

    # client = Client(host="https://pop-os.tailcff25c.ts.net/abcdefghijk")

    stream = client.chat( 
        model = "llama3.2:latest",

        # model = "mistral:7b-instruct", #This has to be added to the hardware we're using.
        messages=[{'role': 'user', 'content': prompt}],
        stream=True,
    )
    completion=""
    try:
        for chunk in stream:
            if chunk is not None:
                if is_verbose is True:
                    print(chunk['message']['content'])
                completion += chunk['message']['content']
                sys.stdout.flush()
                if chunk['message']['content'] == "<|eot_id|>":
                    stream.close() #Close the stream?
                    print("Closing the stream.")

    except Exception as e:
        print(f"{e}") 

    # return request.data["message"
    return completion 


class RAGPipeline(EquoNode):
    
    def __init__(self, model_name="sentence-transformers/multi-qa-MiniLM-L6-cos-v1"):
        self.model = SentenceTransformer(model_name)
        self.documents = []
        self.context = []
        self.entity_index = []
        
    def cosine(self, u: np.ndarray, v: np.ndarray) -> float:
        """
        Cosine similarity metric
        """
        return u.dot(v) / np.sqrt(u.dot(u) * v.dot(v))
    
    def process(self, query: str, documents: list[str]) -> None:
        """
        Order pieces of context by relevance to the user's query
        """
        # Generate embeddings 
        x = self.model.encode(query)
        vectors = self.model.encode(documents)
        self.context = [{"text":doc, "score":self.cosine(vectors[i], x)} for i, doc in enumerate(documents)]
        self.context= sorted(self.context, key=lambda x: x["score"], reverse=True)
        
        
    def retrieve(self, k=10) -> str:
        """
        Retrieve top-K most relevant documents
        Format: 
        Article 1: blah blah blah 
        Article 2: ...
        Article 3: ...
        """
        return "document: ".join([f'{i}: {obj["text"]} 'for i, obj in enumerate(self.context)][:k])
    
    def run(self, query: str, documents: list[str], is_optimized=False) -> str:
        """
            Handle the entire RAG, end-to-end
        """
        pipeline.process(query, documents)
        context = pipeline.retrieve()

        return query_with_ollama(f"Article: {context} \n Answer the following question using the article provided: {query}",
                 is_verbose=False)

    
def parse_pdf(file_path):
    """
    Read the local PDF file
    """
    with open(file_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        
        # Iterate over all the pages and extract text
        for page_num in range(len(reader.pages)):
            page = reader.pages[page_num]
            text += page.extract_text()
            
        return text

In [73]:
pdf_file_path = os.path.join(os.getcwd(), 'smol.pdf')
pdf_text = parse_pdf(pdf_file_path)
document_contents = pdf_text.split(".")
project_name="esg-rag"
equo = equonode(project_name)
db = equonode(None)

In [74]:
# Model for generating Q-A embeddings

pipeline = RAGPipeline()
#Obtain embeddings and convert from ndarray to Python list. 
#Make sure that your embeddings are a list of floating point values before uploading
embeddings = pipeline.model.encode(document_contents)
embeddings = embeddings.tolist()
project_name='esg-report:power-corporation'

### Run the Basic RAG Pipeline

In [79]:
# query = 'How has the leadership of the corporation changed since Paul Desmarais stepped down?'
query = 'Who is currently in charge?'

pipeline = RAGPipeline()

pipeline.process(query, document_contents)
context = pipeline.retrieve()

# answer = query_with_ollama(f"Article: {context} \n Answer the following question using the article provided: {query}",
#                  is_verbose=False)

# answer = pipeline.run(query, document_contents)
print(f"Answer: {answer}")

Answer: Since Paul Desmarais, Jr. stepped down as Chairman and executive officer, André Desmarais, who was previously Deputy Chairman, no longer holds these roles. Additionally, André Desmarais is mentioned to be one of the former executive officers within the past three years, indicating that he has also stepped down from his role.

The article mentions that following Paul Desmarais' retirement, André Desmarais and Pierre Beaudoin are among the Directors who have held management positions in the corporation. However, it does not explicitly state the leadership changes beyond this point.

It is only mentioned that Jeffrey Orr, President and CEO, is an executive officer of the Corporation and therefore not independent. There is no mention of any other changes to the leadership structure since Paul Desmarais stepped down.

However, there have been some notable changes in terms of composition of the Board:

* As of December 31, 2021, there were three women sitting on the Corporation's Boa

### Anonymization API Demo 

#### Here, we are going to keep track of a handful of documents, anonymize their contents, 
#### and keep track of names entities along the way.
#### This allows us to do nearly anything with Generative AI, while remaining compliant in regard to data privacy and standards such as GDPR and SOC-2. 


In [76]:
from dotenv import load_dotenv
# load_dotenv()
# URI = os.getenv("URI")

URI="https://api-equo-ai.tail44cf99.ts.net"

def anonymize(text: str, entities={}, person_count=0, org_count=0, location_count=0) -> str:
    """
    Anonymize your documents using our API for data protection!
    API keys will become available for anyone signing up here:

    https://equo.ai/signup
    """
    r = requests.post(f"{URI}/protect/",
                      json={
                            "text":text,
                            "access_token":"abdefg", 
                            "entities":entities,
                            "item_counts":{
                                "person_count":person_count,
                                "org_count":org_count,
                                "location_count":location_count
                        }
    })
    return r.json()



def track_entities(doc, entities):
    """
    Keep track of previously 
    seen named entities 
    """
    for ent in list(entities.keys()):
    #     print(result["entities"][ent])
        try:
            category = entities[ent]
            doc = doc.replace(ent, category)
        except Exception as e:
            print(e)
    # print(list(result["entities"].keys()))
    return doc 

result = anonymize("Roger and Donald both work for Acme Inc. Their father worked at IBM.")

person_count = result["num_persons"] 
org_count = result["num_orgs"], 
location_count = result["num_locations"]

q2 = "David and Roger are brothers."
print(result["entities"])
entities = result["entities"]

q2 = track_entities(q2, entities)
next_result = anonymize(q2, entities, person_count, org_count, location_count)
print(next_result["text"])

{'Roger': '<PERSON 1>', 'Donald': '<PERSON 2>', 'Acme Inc.': '<ORGANIZATION 1>', 'IBM': '<ORGANIZATION 2>'}
<PERSON 3> and <PERSON 1> are brothers.


In [97]:
entities = {}
person_count = 0
org_count = 0
location_count = 0
anonymized_documents = []

documents = [
    "David and Roger are brothers. Devin is their cousin",
    "Roger and Donald both work for Acme Inc. Their father worked at IBM.",
    "Donald and David are actually best friends. Sometimes they hang out with Roger.",
    "David and Roger are brothers",
    "Donald and Roger sometimes hang out.",
    "David does not talk to his cousin"

]

# context = [obj["text"] for obj in pipeline.context]
for i, doc in enumerate(documents):
# for i, doc in enumerate(context):
# for i, doc in enumerate(document_contents):
    try:
        # Awesome 
#         print(f'Document: {doc}')
        doc = track_entities(doc, entities)
        result = anonymize(doc, entities, person_count, org_count, location_count)
        person_count = result["num_persons"] 
        org_count = result["num_orgs"], 
        location_count = result["num_locations"]
        entities = result["entities"]
        anonymized_documents.append(doc)
        print(f'Document: {doc}')
    except Exception as e:
        print(e)

Document: David and Roger are brothers. Devin is their cousin
Expecting value: line 2 column 1 (char 1)
Document: Donald and <PERSON 1> are actually best friends. Sometimes they hang out with <PERSON 2>.
Document: <PERSON 1> and <PERSON 2> are brothers
Document: <PERSON 4> and <PERSON 2> sometimes hang out.
Document: <PERSON 1> does not talk to his cousin


In [115]:
context = "".join(anonymized_documents[:])
# context = anonymized_documents[-1]
cont = pipeline.context[0]['text']

anon_context = anonymize(cont)
print(anon_context['text'])

 and <PERSON 1> from their executive roles as Co-Chief 
Executive Officers of the <ORGANIZATION 3> on <PERSON 1>, <ORGANIZATION 1> 
is now entirely composed of <ORGANIZATION 2> who are not members of management of the <ORGANIZATION 3>


In [110]:


# question = "Who sometimes hang out together?"
question = "What happened to the leadership of the company after <PERSON 1> stepped down?"
query = f"Article: {anon_context} \n Given the article above, answer the following question: {question}"
ans = query_with_ollama(query, is_verbose=False)

#The LLM is smart enough to keep track of the protected named entities if we are!
print(f"Answer from Protected Data: {ans}")

# print(pipeline.context[0]['text'])
print(anon)
# print(anon_context)

Answer from Protected Data: According to the article, after <PERSON 1> stepped down as Co-Chief Executive Officer of ORGANIZATION 3, the leadership structure changed. Specifically:

* <PERSON 1> is no longer part of the management team.
* The Governance and Nominating Committee (ORGANIZATION 1) is now entirely composed of Directors from ORGANIZATION 2.

It does not provide information about what happened to the leadership of ORGANIZATION 3 after <PERSON 1> stepped down, only that it is no longer being led by management members.
{'text': ' ', 'num_persons': 0, 'num_orgs': 0, 'num_locations': 0, 'entities': {}}


In [105]:
# print(ans)
# print(anon)
print(anon_context, type(anon_context))
# pipeline.context[0]['text']

{'text': ' and <PERSON 1> from their executive roles as Co-Chief \nExecutive Officers of the <ORGANIZATION 3> on <PERSON 1>, <ORGANIZATION 1> \nis now entirely composed of <ORGANIZATION 2> who are not members of management of the <ORGANIZATION 3>', 'num_persons': 1, 'num_orgs': 3, 'num_locations': 0, 'entities': {'André Desmarais': '<PERSON 1>', 'February\xa013, 2020': '<PERSON 1>', 'the Governance and Nominating Committee': '<ORGANIZATION 1>', 'Directors': '<ORGANIZATION 2>', 'Corporation': '<ORGANIZATION 3>'}} <class 'dict'>


## We've learned how to do two things:
#### Setup a RAG Pipeline 
#### Anonymize our Data as a next step toward compliant workflows.

In [96]:
# anonymized_documents[0]
pipeline.context[0]['text']
# # context = "".join(anonymized_documents[2:])
# anon_context = ""
# for doc in anonymized_documents:
#     if entities["Paul Desmarais"] in doc:
#         print(doc)
#         anon_context += doc 
# # context = anonymized_documents[-1]

# question = "How has the leadership changed since <PERSON 10> stepped down?"
# query = f"Article: {anon_context} \n Given the article above, answer the following question: {question}"
# ans = query_with_ollama(query, is_verbose=False)
# print(ans)

' and André Desmarais from their executive roles as Co-Chief \nExecutive Officers of the Corporation on February\xa013, 2020, the Governance and Nominating Committee \nis now entirely composed of Directors who are not members of management of the Corporation'

In [88]:
print(anonymize(pipeline.context[0]['text']))

{'text': ' and <PERSON 1> from their executive roles as Co-Chief \nExecutive Officers of the <ORGANIZATION 3> on <PERSON 1>, <ORGANIZATION 1> \nis now entirely composed of <ORGANIZATION 2> who are not members of management of the <ORGANIZATION 3>', 'num_persons': 1, 'num_orgs': 3, 'num_locations': 0, 'entities': {'André Desmarais': '<PERSON 1>', 'February\xa013, 2020': '<PERSON 1>', 'the Governance and Nominating Committee': '<ORGANIZATION 1>', 'Directors': '<ORGANIZATION 2>', 'Corporation': '<ORGANIZATION 3>'}}


In [67]:
for doc in anonymized_documents:

        print(doc)

<PERSON 1>3<PERSON <PERSON 10>> <PERSON 1>
